# ICP For Point-Cloud Registration

Implementing the ICP algorithm to register two point clouds. 
- Registration refers to aligning to be coherent. 
- In robotics, when lidar scans from different parts of a room, for instance, need to be joined together to create a full map of the environment.
- Some examples in [this paper](http://redwood-data.org/indoor_lidar_rgbd/paper.pdf).

## Setup 

- Notebook from [CS237a hw3, part3](https://colab.research.google.com/drive/1O94OB54oYSu7E1CAxP29IeuCnfJnA3t8?usp=sharing)
- Use the [Stanford Bunny](https://graphics.stanford.edu/data/3Dscanrep/) dataset
- Referring to partial point cloud, $A$, as the "source", 
- Referring to full scan, $B$, of the bunny as the "target"

```python
cloud_A = o3d.io.read_point_cloud("data/bun045.ply")
cloud_B = o3d.io.read_point_cloud("data/bun_zipper.ply")
```

In [2]:
#-----------------------------------+
# In terminal, create a virtual env #
#-----------------------------------+
# uv venv
# source .venv/bin/activate
#
#-----------------------------------+
# and install dependencies          #
#-----------------------------------+
# uv pip install open3d 
# uv pip install ipykernel
# uv pip install ipywidgets
# 
#-----------------------------------+
# launch vs code                    #
#-----------------------------------+
# code .                            #
#-----------------------------------+ 
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from utils import *

cloud_A = o3d.io.read_point_cloud("data/bun045.ply")
cloud_B = o3d.io.read_point_cloud("data/bun_zipper.ply")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Visualize point cloud data

- Use the slider bars to view the bunny from different angles.
- Partial point cloud $A$ scan is not a complete scan of the bunny
- We will be referring to this partial point cloud, $A$, as the "source", 
- We will be referring to the full scan $B_t$ of the bunny as the "target"

In [3]:
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_A]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

In [4]:
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_B]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

In [5]:
# View both clouds A (partial_scan) and B (full_bunny)
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_A, cloud_B]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

# Global Registration

Our goal will be to match the point cloud data
- containing a partial view of the bunny (A)
- with the full 3D mesh model of the same bunny (B)

We will do this using the RANSAC algorithm.


$$ 
\text{Given A, B} \\
\text{find a matrix} \quad T \\
\text{s.t.} \quad B \approx A T 
$$

## Extract features and downsample
We can see that the two point clouds are clearly misaligned right now. 

- So first, we will extract some features of each point cloud. 
- These features, called the "FPFH" features of the point clouds are a vector of 33 values for each point in the point cloud that represents some unique features of that point. 
- Therefore, if we have N points in point cloud, your FPFH feature matrix for that point cloud will be of shape (N, 33).

Since N can be large for raw point cloud data, we will downsample it a bit so that it is easier to experiment with.

In [6]:
# Extract features
def preprocess_point_cloud(point_cloud, voxel_size):
    print(":: Downsample with a voxel size %.3f." % voxel_size)
    point_cloud_sample = point_cloud.voxel_down_sample(voxel_size)

    radius_normal = voxel_size * 2
    print(":: Estimate normal with search radius %.3f." % radius_normal)
    point_cloud_sample.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))

    radius_feature = voxel_size * 5
    print(":: Compute FPFH feature with search radius %.3f." % radius_feature)
    point_cloud_features = o3d.pipelines.registration.compute_fpfh_feature(
        point_cloud_sample,
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100))
    
    return point_cloud_sample, point_cloud_features

def prepare_dataset(cloud_A, cloud_B, voxel_size):
    A_sample, A_features = preprocess_point_cloud(cloud_A, voxel_size)
    B_sample, B_features = preprocess_point_cloud(cloud_B, voxel_size)
    return cloud_A, cloud_B, A_sample, B_sample, A_features, B_features

cloud_A, cloud_B, A_sample, B_sample, A_features, B_features = prepare_dataset(cloud_A, cloud_B, 0.01)

:: Downsample with a voxel size 0.010.
:: Estimate normal with search radius 0.020.
:: Compute FPFH feature with search radius 0.050.
:: Downsample with a voxel size 0.010.
:: Estimate normal with search radius 0.020.
:: Compute FPFH feature with search radius 0.050.


In [7]:
# RANSAC algorithm using open3d to estimate T = [R t]
def register(A_sample, B_sample, A_features, B_features, voxel_size):
    distance_threshold = voxel_size * 1.5
    
    # Open3D’s global registration function
    # Estimate a rigid transform between two point clouds using feature matches FPFH + RANSAC
    # Works even when clouds are far apart initially 
    # Good first stage before local refinement with ICP
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        A_sample,
        B_sample,
        A_features,
        B_features,
        False,
        distance_threshold
    )

    return result.transformation, result

T, result = register(A_sample, B_sample, A_features, B_features, 0.01)
print(f":: Transformation T :: \n {T}")
print(result)

:: Transformation T :: 
 [[ 0.79740517  0.01188526  0.60332722 -0.05877124]
 [-0.00501994  0.99990208 -0.01306285 -0.00893943]
 [-0.6034234   0.00738772  0.79738675 -0.01871012]
 [ 0.          0.          0.          1.        ]]
RegistrationResult with fitness=1.000000e+00, inlier_rmse=8.234799e-03, and correspondence_set size of 371
Access transformation to get result.


# After RANSAC
Let's now apply this transformation matrix to our point cloud data to see how well the point clouds are now registered.

In [8]:
from copy import deepcopy 
A_transformed = deepcopy(cloud_A).transform(T)
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_B, A_transformed]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

As you can see, `A.transform(T)` and `clound B: full_bunny` are much better aligned than before but need fine-tuning.

We will now use ICP to try to improve this alignment, a process also called "Local Refinement"

# Basic ICP

1. For every point in A, find its nearest neighbor in B
2. Find matrix T that maps points in A to B 
3. Apply this transformation, T, to A
4. Compute the average error for this transformation across all points in A
5. Repeat steps 1-4 for N iterations, or break if:

$abs(e_t - e_{t-1}) < \tau$

where $e_i$ is the average error in the i-th iteration and $\tau$ is the tolerance set as a hyperparameter


In [9]:
from sklearn.neighbors import NearestNeighbors
from tqdm import trange 

def best_rigid_transform(A, B):
    # A.shape = (371, 3) (N, 3)
    # B.shape = (795, 3) (M, 3)

    # Compute centroids
    Ac = A.mean(axis=0)
    Bc = B.mean(axis=0)

    # Center the points
    AA = A - Ac
    BB = B - Bc

    # Cross-covariance
    H = AA.T @ BB 

    # SVD
    U, _, Vt = np.linalg.svd(H)

    # Rotation 
    R = Vt.T @ U.T 

    if np.linalg.det(R) < 0:
        Vt[-1] = -Vt[-1]
        R = Vt.T @ U.T 
    
    # find t = Bc - R*Ac
    t = Bc.T - R @ Ac.T

    m = A.shape[1]
    T = np.identity(m+1)

    T[:m, :m] = R 
    T[:m,  m] = t 
    return T, R, t

def nn(A, B, radius=0.01):
    neigh = NearestNeighbors(n_neighbors=1)
    neigh.fit(B)
    distances, indices = neigh.kneighbors(A, return_distance=True)
    return distances.ravel(), indices.ravel()

def icp(A, B, max_iters=20, tol=0.001, knn_radius=0.01):
    m = A.shape[1]
    
    src = np.ones((m+1, A.shape[0]))
    dst = np.ones((m+1, B.shape[0]))

    src[:m, :] = np.copy(A.T)
    dst[:m, :] = np.copy(B.T)

    prev_error = 0
    for i in trange(max_iters):
        distances, indices = nn(src[:m, :].T, dst[:m, :].T, radius=knn_radius)
        T_est, _, _ = best_rigid_transform(src[:m, :].T, dst[:m, indices].T)
        src = T_est @ src 

        avg_error = np.mean(distances)
        if np.abs(avg_error - prev_error) < tol:
            print(f":: Converged at iteration {i}")
            break
        prev_error = avg_error
    
    T, _, _ = best_rigid_transform(A, src[:m, :].T)
    return T 

T_icp = icp(np.asarray(A_sample.points), np.asarray(B_sample.points), max_iters=20, tol=1e-6)

# Comparing T_icp with T 
print(np.allclose(T, T_icp))
print(f":: Transformation T :: \n {T}")
print(f":: Transformation T :: \n {T_icp}")

A_transformed_icp = deepcopy(cloud_A).transform(T_icp)
widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_B, A_transformed_icp]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation")
)

 85%|████████▌ | 17/20 [00:00<00:00, 1376.14it/s]

:: Converged at iteration 17
False
:: Transformation T :: 
 [[ 0.79740517  0.01188526  0.60332722 -0.05877124]
 [-0.00501994  0.99990208 -0.01306285 -0.00893943]
 [-0.6034234   0.00738772  0.79738675 -0.01871012]
 [ 0.          0.          0.          1.        ]]
:: Transformation T :: 
 [[ 0.8229891  -0.00142242  0.56805538 -0.05273214]
 [-0.00494844  0.99994097  0.00967308 -0.00129893]
 [-0.56803561 -0.01077182  0.82293348 -0.01079954]
 [ 0.          0.          0.          1.        ]]


interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

# Robust ICP Using open3D

In [10]:
def register_robust(cloud_A, cloud_B, T, threshold=0.01):
    result = o3d.pipelines.registration.registration_icp(
        cloud_A,
        cloud_B,
        threshold,
        T,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=3000)
    )

    T_robust = result.transformation
    return T_robust

T_robust = register_robust(cloud_A, cloud_B, T, 0.00001)

A_transformed_robust = deepcopy(cloud_A).transform(T_robust)

widgets.interact(
    plot_point_clouds,
    pcd_list=widgets.fixed([cloud_B, A_transformed_robust]),
    azim=widgets.IntSlider(-90, min=-180, max=180, step=5, description="Azimuth"),
    elev=widgets.IntSlider(90, min=0, max=90, step=5, description="Elevation"))

interactive(children=(IntSlider(value=-90, description='Azimuth', max=180, min=-180, step=5), IntSlider(value=…

<function utils.plot_point_clouds(pcd_list, azim=-60, elev=30)>

In [17]:
print(f"T == T_robust: {np.allclose(T_robust, T)}")
print(f"T_icp \n {T_icp}")
print(f"T_robust \n {T_robust}")

T == T_robust: True
T_icp 
 [[ 0.8229891  -0.00142242  0.56805538 -0.05273214]
 [-0.00494844  0.99994097  0.00967308 -0.00129893]
 [-0.56803561 -0.01077182  0.82293348 -0.01079954]
 [ 0.          0.          0.          1.        ]]
T_robust 
 [[ 0.79740517  0.01188526  0.60332722 -0.05877124]
 [-0.00501994  0.99990208 -0.01306285 -0.00893943]
 [-0.6034234   0.00738772  0.79738675 -0.01871012]
 [ 0.          0.          0.          1.        ]]
